In [1]:
# ============================================================
# 🧠 KNOWLEDGE GAP DETECTOR
# Text and Speech Analysis / NLP Application
# Google Colab - Single Cell
# ============================================================

!pip -q install gradio

import gradio as gr
import re

# ============================================================
# KNOWLEDGE BASE
# ============================================================

knowledge_base = {
    "python": {
        "concepts": [
            "variables",
            "data types",
            "conditional statements",
            "loops",
            "functions",
            "lists",
            "dictionaries",
            "classes",
            "exceptions"
        ]
    },

    "html": {
        "concepts": [
            "html structure",
            "headings",
            "paragraphs",
            "links",
            "images",
            "lists",
            "tables",
            "forms",
            "input elements"
        ]
    },

    "css": {
        "concepts": [
            "selectors",
            "colors",
            "fonts",
            "margin",
            "padding",
            "borders",
            "box model",
            "flexbox",
            "grid"
        ]
    },

    "computer networks": {
        "concepts": [
            "network",
            "osi model",
            "tcp",
            "udp",
            "ip address",
            "router",
            "switch",
            "http",
            "dns"
        ]
    },

    "machine learning": {
        "concepts": [
            "dataset",
            "features",
            "training",
            "testing",
            "classification",
            "regression",
            "supervised learning",
            "unsupervised learning",
            "model evaluation"
        ]
    },

    "natural language processing": {
        "concepts": [
            "tokenization",
            "stopwords",
            "stemming",
            "lemmatization",
            "part of speech",
            "named entity recognition",
            "sentiment analysis",
            "language model"
        ]
    }
}


# ============================================================
# NORMALIZE TEXT
# ============================================================

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ============================================================
# FIND KNOWLEDGE GAPS
# ============================================================

def detect_gaps(topic, answer):

    if not topic or not answer:
        return (
            "⚠️ Please enter both the topic and student's answer.",
            ""
        )

    topic_clean = normalize_text(topic)

    # Find matching topic
    selected_topic = None

    for key in knowledge_base:

        if key in topic_clean or topic_clean in key:
            selected_topic = key
            break

    if selected_topic is None:

        return (
            "❌ Topic not available in the demo knowledge base.",
            """
Available topics:

• Python
• HTML
• CSS
• Computer Networks
• Machine Learning
• Natural Language Processing
"""
        )

    concepts = knowledge_base[selected_topic]["concepts"]

    answer_clean = normalize_text(answer)

    # --------------------------------------------------------
    # CHECK EACH CONCEPT
    # --------------------------------------------------------

    understood = []
    missing = []

    for concept in concepts:

        concept_clean = normalize_text(concept)

        # Check exact concept
        if concept_clean in answer_clean:
            understood.append(concept)

        else:

            # Check individual words
            concept_words = concept_clean.split()

            matches = 0

            for word in concept_words:

                if word in answer_clean:
                    matches += 1

            if matches >= len(concept_words):
                understood.append(concept)
            else:
                missing.append(concept)

    # --------------------------------------------------------
    # CALCULATE COVERAGE
    # --------------------------------------------------------

    total = len(concepts)

    coverage = (len(understood) / total) * 100

    # --------------------------------------------------------
    # LEVEL
    # --------------------------------------------------------

    if coverage >= 80:

        level = "🟢 GOOD UNDERSTANDING"

        recommendation = (
            "You covered most important concepts. "
            "Review the missing concepts once more."
        )

    elif coverage >= 50:

        level = "🟡 PARTIAL UNDERSTANDING"

        recommendation = (
            "You understand some important concepts, "
            "but several areas need more study."
        )

    else:

        level = "🔴 KNOWLEDGE GAP DETECTED"

        recommendation = (
            "Several important concepts are missing. "
            "Study the missing topics and try the test again."
        )

    # --------------------------------------------------------
    # BUILD OUTPUT
    # --------------------------------------------------------

    result = f"""
🧠 KNOWLEDGE GAP ANALYSIS

📚 Topic:
{selected_topic.title()}

🎯 Understanding Level:
{level}

📊 Concept Coverage:
{coverage:.1f}%

✅ Concepts Detected:
"""

    if understood:

        for item in understood:
            result += f"• {item.title()}\n"

    else:
        result += "None detected\n"

    result += "\n❌ Possible Knowledge Gaps:\n"

    if missing:

        for item in missing:
            result += f"• {item.title()}\n"

    else:

        result += "No major gaps detected 🎉\n"

    result += f"""

💡 Recommendation:
{recommendation}
"""

    # --------------------------------------------------------
    # STUDY PLAN
    # --------------------------------------------------------

    study_plan = ""

    if missing:

        study_plan = "📖 SUGGESTED STUDY PLAN\n\n"

        for i, item in enumerate(missing, 1):

            study_plan += (
                f"{i}. Study: {item.title()}\n"
            )

    else:

        study_plan = (
            "🎉 Excellent! You covered all the "
            "important concepts in this knowledge base."
        )

    return result, study_plan


# ============================================================
# GRADIO APPLICATION
# ============================================================

with gr.Blocks(
    title="Knowledge Gap Detector"
) as app:

    gr.Markdown(
        """
        # 🧠 Knowledge Gap Detector

        ### Find which important concepts are missing from your answer.

        **Topic + Student Answer → NLP Analysis → Concept Detection → Knowledge Gaps**
        """
    )

    gr.Markdown(
        """
        ### 📌 Available Demo Topics

        🐍 Python
        🌐 HTML
        🎨 CSS
        🌐 Computer Networks
        🤖 Machine Learning
        💬 Natural Language Processing
        """
    )

    # --------------------------------------------------------
    # INPUTS
    # --------------------------------------------------------

    topic_input = gr.Textbox(
        label="📚 Study Topic",
        placeholder="Example: Python",
        lines=1
    )

    answer_input = gr.Textbox(
        label="✍️ Student Answer",
        placeholder="Enter what you know about the topic...",
        lines=10
    )

    # --------------------------------------------------------
    # BUTTON
    # --------------------------------------------------------

    detect_button = gr.Button(
        "🔍 Detect Knowledge Gaps",
        variant="primary"
    )

    # --------------------------------------------------------
    # OUTPUTS
    # --------------------------------------------------------

    result_output = gr.Textbox(
        label="📊 Knowledge Gap Analysis",
        lines=20
    )

    study_output = gr.Textbox(
        label="📖 Suggested Study Plan",
        lines=10
    )

    # --------------------------------------------------------
    # BUTTON ACTION
    # --------------------------------------------------------

    detect_button.click(
        fn=detect_gaps,
        inputs=[
            topic_input,
            answer_input
        ],
        outputs=[
            result_output,
            study_output
        ]
    )

    # --------------------------------------------------------
    # EXAMPLE
    # --------------------------------------------------------

    gr.Examples(
        examples=[
            [
                "Python",
                "Python has variables, data types, loops and functions. "
                "Lists are used to store multiple values."
            ],
            [
                "HTML",
                "HTML has headings, paragraphs, links and images. "
                "We can create lists using HTML."
            ],
            [
                "Machine Learning",
                "Machine learning uses datasets and features. "
                "Training is used to build a model."
            ]
        ],
        inputs=[
            topic_input,
            answer_input
        ]
    )

    gr.Markdown(
        """
        ---

        ### ✨ Features

        📝 Student answer analysis
        🧠 Concept detection
        🔍 Missing concept identification
        📊 Knowledge coverage percentage
        📖 Suggested study topics
        🎯 Understanding-level prediction
        🌐 Interactive Gradio interface

        **Text and Speech Analysis Project**
        """
    )


# ============================================================
# RUN APPLICATION
# ============================================================

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9de5ce395b0924f643.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
